# 24 - Behavior Cloning With A NumPy MLP

## What / Why / How

**What we are trying to do:** Train a small neural behavior cloning policy from scratch using NumPy.

**Why this matters:** Frameworks are useful, but implementing a tiny network once makes policy learning and gradients more concrete.

**How we will do it:** Generate supervised reaching data, train a one-hidden-layer MLP by backpropagation, then evaluate it in closed-loop rollout.

## Goal

Train a small behavior cloning policy from scratch using NumPy.

This avoids deep-learning framework setup while teaching:

- Mini-batch training
- Nonlinear policies
- Train/validation error

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(24)
N = 1200
state = rng.uniform(-1.5, 1.5, size=(N, 2))
target = rng.uniform(-1.0, 1.0, size=(N, 2))
obs = np.c_[state, target]
action = np.tanh(1.3 * (target - state)) * 0.25
action += rng.normal(0, 0.01, size=action.shape)

idx = rng.permutation(N)
train_idx, val_idx = idx[:900], idx[900:]
X_train, Y_train = obs[train_idx], action[train_idx]
X_val, Y_val = obs[val_idx], action[val_idx]
print(X_train.shape, Y_train.shape)

In [ ]:
hidden = 32
W1 = rng.normal(0, 0.2, size=(4, hidden))
b1 = np.zeros(hidden)
W2 = rng.normal(0, 0.2, size=(hidden, 2))
b2 = np.zeros(2)

def forward(X):
    Z1 = X @ W1 + b1
    H = np.tanh(Z1)
    Y = H @ W2 + b2
    return Z1, H, Y

lr = 0.03
batch = 64
for epoch in range(250):
    batch_idx = rng.choice(len(X_train), size=batch, replace=False)
    Xb, Yb = X_train[batch_idx], Y_train[batch_idx]
    Z1, H, pred = forward(Xb)
    grad_pred = 2 * (pred - Yb) / len(Xb)
    grad_W2 = H.T @ grad_pred
    grad_b2 = grad_pred.sum(axis=0)
    grad_H = grad_pred @ W2.T
    grad_Z1 = grad_H * (1 - np.tanh(Z1) ** 2)
    grad_W1 = Xb.T @ grad_Z1
    grad_b1 = grad_Z1.sum(axis=0)
    W2 -= lr * grad_W2
    b2 -= lr * grad_b2
    W1 -= lr * grad_W1
    b1 -= lr * grad_b1

_, _, train_pred = forward(X_train)
_, _, val_pred = forward(X_val)
print("train MSE:", np.mean((train_pred - Y_train)**2))
print("val MSE:", np.mean((val_pred - Y_val)**2))

In [ ]:
def policy(pos, goal):
    _, _, out = forward(np.array([[pos[0], pos[1], goal[0], goal[1]]]))
    return np.clip(out[0], -0.25, 0.25)

pos = np.array([-1.2, 0.8])
goal = np.array([0.8, -0.7])
route = [pos.copy()]
for _ in range(35):
    pos = pos + policy(pos, goal)
    route.append(pos.copy())
route = np.array(route)
print("final error:", np.linalg.norm(route[-1] - goal))

In [ ]:
if HAS_PLOT:
    plt.figure(figsize=(5, 5))
    plt.plot(route[:, 0], route[:, 1], marker="o", markersize=2)
    plt.scatter([goal[0]], [goal[1]], c="tab:red")
    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.title("NumPy behavior cloning rollout")
    plt.show()
else:
    plot_unavailable()

## Exercises

1. Add a second hidden layer.
2. Train with fewer samples and observe overfitting.
3. Add action delay to closed-loop evaluation.